<div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);padding:32px 28px;border-radius:12px;margin-bottom:8px">
<h1 style="color:#e9c46a;margin:0 0 6px">🤖 FIFA World Cup 2026</h1>
<h2 style="color:#a8dadc;font-weight:400;margin:0 0 14px">Notebook 3 of 4 — Modeling & Evaluation</h2>
<p style="color:#ccc;line-height:1.6;margin:0 0 16px">
We load the pre-built feature matrix, train 4 classifiers on pre-2022 data,
evaluate on a held-out 2022+ test set, and select the best model for prediction.
</p>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#2a9d8f">🤖 ML</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#e76f51;margin-left:6px">4 Models</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#457b9d;margin-left:6px">Time-based split</span>
<span style="padding:4px 12px;border-radius:20px;font-size:.85em;color:white;background:#6a4c93;margin-left:6px">💾 Saves best model</span>
</div>

---
### What we do here
1. Load features from `features/historical_features.csv`
2. Engineer derived features (`elo_diff`, `form_wr_diff` etc.)
3. Time-based train/test split — train pre-2022, test 2022+
4. Train: Logistic Regression, Random Forest, Gradient Boosting, HistGradBoost
5. Evaluate: accuracy, macro F1, log loss, confusion matrix, ROC curves, feature importance
6. Save best model + scaler for notebook 04

> **Prerequisite:** Run `02_feature_engineering.ipynb` first


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                               HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, f1_score, log_loss,
                              classification_report, confusion_matrix,
                              roc_curve, auc)

sns.set_style("whitegrid")
PALETTE = ["#2a9d8f","#e9c46a","#e76f51","#264653","#a8dadc",
           "#457b9d","#fca311","#6a4c93","#f4a261","#1d3557"]
print("✅ Imports ready")

## 2. Load Features

In [ ]:
feat_df = pd.read_csv("features/historical_features.csv")
feat_df["date"] = pd.to_datetime(feat_df["date"])
print(f"Feature matrix: {feat_df.shape[0]:,} rows × {feat_df.shape[1]} cols")
feat_df.tail(3)

## 3. Build Modeling Dataset

In [ ]:
FEATURES = [
    "home_elo_pre", "away_elo_pre", "elo_diff",
    "home_form_wr", "away_form_wr", "form_wr_diff",
    "home_form_gf", "home_form_ga",
    "away_form_gf", "away_form_ga",
    "home_form_comp", "away_form_comp",
    "h2h_home_wr", "h2h_away_wr",
    "is_neutral", "is_wc",
]

mdf = feat_df[feat_df["date"] >= "1990-01-01"].copy()
mdf["result"]       = np.where(mdf["home_score"]>mdf["away_score"],"Home Win",
                      np.where(mdf["home_score"]<mdf["away_score"],"Away Win","Draw"))
mdf["elo_diff"]     = mdf["home_elo_pre"] - mdf["away_elo_pre"]
mdf["form_wr_diff"] = mdf["home_form_wr"] - mdf["away_form_wr"]
mdf["is_neutral"]   = mdf["neutral"].astype(int)
mdf["is_wc"]        = mdf["tournament"].str.contains("World Cup", na=False).astype(int)
mdf = mdf[FEATURES + ["result","date"]].dropna()

cutoff = pd.Timestamp("2022-01-01")
train  = mdf[mdf["date"] <  cutoff]
test   = mdf[mdf["date"] >= cutoff]
X_tr, y_tr = train[FEATURES].values, train["result"].values
X_te, y_te = test[FEATURES].values,  test["result"].values
scaler = StandardScaler().fit(X_tr)

print(f"Total samples    : {len(mdf):,}")
print(f"Train (pre-2022) : {len(X_tr):,}")
print(f"Test  (2022+)    : {len(X_te):,}")
print("\nClass balance (train):")
print(pd.Series(y_tr).value_counts(normalize=True).round(3))

## 4. Train 4 Classifiers

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=0.5, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=500, max_depth=10,
                                                   min_samples_leaf=15, random_state=42, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingClassifier(n_estimators=400, max_depth=4,
                                                       learning_rate=0.04, random_state=42),
    "HistGradBoost"      : HistGradientBoostingClassifier(max_iter=600, max_depth=5,
                                                           learning_rate=0.04, random_state=42),
}

trained, results_eval, classes = {}, {}, None
for name, mdl in models.items():
    Xtr = scaler.transform(X_tr) if name=="Logistic Regression" else X_tr
    Xte = scaler.transform(X_te) if name=="Logistic Regression" else X_te
    mdl.fit(Xtr, y_tr)
    preds = mdl.predict(Xte)
    probs = mdl.predict_proba(Xte)
    if classes is None: classes = mdl.classes_
    trained[name] = mdl
    results_eval[name] = {
        "acc":accuracy_score(y_te,preds), "f1":f1_score(y_te,preds,average="macro"),
        "ll":log_loss(y_te,probs,labels=classes), "preds":preds, "probs":probs,
    }
    print(f"  ✓ {name:25s}  acc={results_eval[name]['acc']:.3f}  "
          f"f1={results_eval[name]['f1']:.3f}  log_loss={results_eval[name]['ll']:.3f}")

metrics_df = pd.DataFrame([
    {"Model":n,"Accuracy":round(v["acc"],3),"Macro F1":round(v["f1"],3),"Log Loss":round(v["ll"],3)}
    for n,v in results_eval.items()
]).sort_values("Macro F1", ascending=False).reset_index(drop=True)

best_name  = metrics_df.iloc[0]["Model"]
best_model = trained[best_name]
print(f"\n🏅 Best model: {best_name}")
metrics_df

## 5. Model Comparison Visualizations

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs  = fig.add_gridspec(3, 4, height_ratios=[0.8, 1, 1], hspace=0.45, wspace=0.35)

# Row 1: metric bars
ax = fig.add_subplot(gs[0, :])
x, w = np.arange(len(metrics_df)), 0.25
ax.bar(x-w, metrics_df["Accuracy"], w, label="Accuracy", color=PALETTE[0])
ax.bar(x,   metrics_df["Macro F1"], w, label="Macro F1", color=PALETTE[1])
ax2 = ax.twinx()
ax2.bar(x+w, metrics_df["Log Loss"], w, label="Log Loss", color=PALETTE[2], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metrics_df["Model"], fontsize=11)
ax.set_ylabel("Accuracy / Macro F1 (↑ better)", fontsize=11)
ax2.set_ylabel("Log Loss (↓ better)", fontsize=11)
ax.set_title("Model Comparison — Held-out 2022+ Test Set", fontsize=13, fontweight="bold")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

# Row 2: confusion matrices
for i, (name, r) in enumerate(results_eval.items()):
    ax = fig.add_subplot(gs[1, i])
    cm = confusion_matrix(y_te, r["preds"], labels=classes, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=classes, yticklabels=classes,
                cbar=False, ax=ax, linewidths=0.5, linecolor="white")
    ax.set_title(f"{name}\nacc={r['acc']:.2f} | f1={r['f1']:.2f}", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True" if i==0 else "")

# Row 3: feature importance + ROC
ax = fig.add_subplot(gs[2, :2])
rf  = trained["Random Forest"]
imp = pd.DataFrame({"feature":FEATURES,"importance":rf.feature_importances_}).sort_values("importance")
ax.barh(imp["feature"], imp["importance"], color=PALETTE[3])
ax.set_title("Feature Importance (Random Forest)", fontsize=11, fontweight="bold")
ax.set_xlabel("Importance")

ax    = fig.add_subplot(gs[2, 2:])
y_bin = label_binarize(y_te, classes=classes)
best_r = results_eval[best_name]
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:,i], best_r["probs"][:,i])
    ax.plot(fpr, tpr, linewidth=2.5, label=f"{cls}  AUC={auc(fpr,tpr):.2f}")
ax.plot([0,1],[0,1],"k--", alpha=0.4, label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curves — {best_name}", fontsize=11, fontweight="bold")
ax.legend(loc="lower right")

plt.suptitle("Model Evaluation Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.show()

## 6. Detailed Classification Report

In [ ]:
print(f"=== {best_name} — Held-out 2022+ test set ===")
print(f"Accuracy : {results_eval[best_name]['acc']:.4f}")
print(f"Macro F1 : {results_eval[best_name]['f1']:.4f}")
print(f"Log Loss : {results_eval[best_name]['ll']:.4f}\n")
print(classification_report(y_te, results_eval[best_name]["preds"], digits=3))

## 7. Save Best Model

In [ ]:
import os, pickle
os.makedirs("features", exist_ok=True)

with open("features/best_model.pkl","wb") as f:
    pickle.dump({"model":best_model, "name":best_name,
                 "classes":classes, "scaler":scaler, "features":FEATURES}, f)
print(f"✅ Saved: features/best_model.pkl  (model: {best_name})")

---
## ✅ Modeling Complete

| Model | Accuracy | Macro F1 | Log Loss |
|-------|----------|----------|----------|
| Best | see output | see output | see output |

**Why this is a hard problem:** Predicting football is inherently noisy — upsets happen all the time. A 3-class accuracy of ~56% on held-out real matches (2022+) is meaningfully better than random (33%) and the naive home-win baseline (~47%).

**→ Next notebook:** `04_prediction.ipynb`
